<a href="https://colab.research.google.com/github/nicolebid/birthday-paradox/blob/main/birthday_paradox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, timedelta
from collections import Counter

#**The Birthday Paradox** 🎂

## **The Problem**

TikTok's fastest-rising creator, Evan Odds, is gearing up for his birthday. His weekly livestream challenges&#8212;always with a cash prize&#8212;have viewers coming back, hungry for more. This week, he's promising his biggest challenge yet: The Birthday Paradox.

It's a simple game. One by one, Evan will pull random viewers out of the chat and into the call, asking each one the same question: "When is your birthday?" The moment two guests share a birthday, the game ends, and everyone in the chat who guessed the exact number of guests it would take splits the cash prize! 💸

The chat immediately erupts with guesses. Some say 50. Some say 100. One viewer swears it'll take "at least 200, easy." Evan smiles and hits "Go Live."

*How many would you guess?*


## **The Big Question**
Viewers are betting on the exact number of guests it'll take in one round of the game&#8212;a not-so-easy task. This number changes each round. Some rounds could end after a handful of guests, while others could take much longer. One approach is finding the *tipping point*, a point where the chance of a shared birthday becomes more likely than not (greater than 50%). The underlying question we will solve is:  

>**How many guests need to join the call so that it's more likely than not that at least two guests share a birthday?**

Before we solve this problem, let's explore with experimentation! Try out the simulated game below. See if you can narrow down a range of values for where the tipping point would fall.


In [ ]:
tally = Counter()

def simulate_game():
    day_list = []
    current_bday = random.randint(1, 365)
    match = False
    count = 1

    while not match:
        if current_bday in day_list:
            match = True
        else:
            day_list.append(current_bday)
            current_bday = random.randint(1, 365)
            count += 1

    day_list.append(current_bday)
    matched_position = day_list.index(current_bday)
    match_date = datetime(2023, 1, 1) + timedelta(days=current_bday - 1)
    return count, day_list, matched_position, match_date

def bday_game():
    count, day_list, matched_position, match_date = simulate_game()

    guest_numbers = list(range(1, count + 1))
    birthdays = [(datetime(2023, 1, 1) + timedelta(days=d - 1)).strftime('%B %d') for d in day_list]
    df = pd.DataFrame({"Guest": guest_numbers, "Birthday": birthdays})

    def highlight_match(row):
        if row.name == matched_position or row.name == len(day_list) - 1:
            return ['background-color:#5dade2'] * len(row)
        return [''] * len(row)

    display(df.style
        .apply(highlight_match, axis=1)
        .hide(axis="index")
        .set_table_styles([
            {'selector': 'th, td', 'props': [('font-size', '9pt'), ('padding', '2px 6px')]}
        ])
    )
    print(f"\nMatch Found after {count} guests!\nGuest {matched_position + 1} and Guest {count} share a birthday on {match_date.strftime('%B %d')}.")
    return count

def plot_tally():
  if not tally:
    return
  else:
    xs = sorted(tally.keys())
    ys = [tally[x] for x in xs]
    plt.figure(figsize=(10, 3))
    plt.bar(xs, ys)

    x_max = max(40, max(xs) + 5)
    y_max = max(10, max(ys) + 1)

    plt.xlim(0, x_max)
    plt.ylim(0, y_max)

    x_step = 1 if x_max <= 40 else 2
    y_step = max(1, y_max // 10)

    plt.xticks(range(0, x_max + 1, x_step))
    plt.yticks(range(0, y_max + 1, y_step))

    plt.xlabel("Number of Guests Needed for a Match")
    plt.ylabel("Number of Games")
    plt.title("Outcomes Across all Games Played")
    plt.show()

# Game buttons
start_button = widgets.Button(description="Start Game!", layout=widgets.Layout(width='100px'))
reset_button = widgets.Button(description="Reset", layout=widgets.Layout(width='100px'))
output = widgets.Output()

def on_start_clicked(b):
    with output:
        output.clear_output()
        result = bday_game()
        tally[result] += 1
        plot_tally()

def on_reset_clicked(b):
    tally.clear()
    with output:
        output.clear_output()

start_button.on_click(on_start_clicked)
reset_button.on_click(on_reset_clicked)
button_row = widgets.HBox([start_button, reset_button])
display(button_row, output)





Output()

## **Game Analysis**

After playing several rounds of the game, you might've developed a gut feeling about the tipping point. But is your intuition correct? Let's break down this problem using probability techniques to find out! 🔍




### What are we solving for?

To find the probability that at least 2 guests share a birthday, $ P(\text{at least 1 match})$, we will use the **complement rule**, since it's easier to calculate the probability that no one shares a birthday, $P(\text{no match})$, and then subtract from 1.

$$P(\text{at least 1 match}) = 1 − P(\text{no match})$$

### What are the assumptions?

For simplicity, assume it's equally likely to be born on any of the 365 days in a year, ignore leap years, and assume each guest's birthday is independent of the others.



### What is the pattern?

Let's walk through the game, guest by guest, and determine probability that each guest doesn't share a birthday with the others.

Let $P(G_N)$ be the probability that Guest $N$ does not share a birthday with any of the previous guests, and consider the conditional probabilities, as follows:

* **Guest 1** - Their birthday will be one of the 365 days, with no other birthdays to match with: $P(G_1) = 1$
* **Guest 2** - For there to be no match, their birthday can't land on the same day as Guest 1's. There are 364/365 days it could fall on:  $P(G_2 | G_1) = \frac{364}{365}$
* **Guest 3** - Given that Guests 1 and 2 didn't share a birthday, there are 2 days Guest 3's birthday must avoid: $P(G_3 | G_1 \cap G_2) = \frac{365−2}{365} = \frac{363}{365}$
* **Guest 4** - Given that Guests 1, 2 and 3, didn't share a birthday, there are 3 days Guest 4's birthday must avoid:  $P(G_4 | G_1 \cap G_2 \cap G_3) = \frac{365−3}{365} = \frac{362}{365}$

Continuing this pattern, we generalize to $N$ number of guests:

$$P(G_N | G_1\cap G_2\cap ... \cap G_{N-1}) = \frac{365 - (N - 1)}{365} = \frac{366 - N}{365}$$

To find $P(\text{no match})$, we apply the chain rule for conditional probability as follows:

$$
\begin{aligned}
P(\text{no match}) &= P(G_1 \cap G_2 \cap \cdots \cap G_N) \\
&= P(G_1) \cdot P(G_2 \mid G_1) \cdot P(G_3 \mid G_1 \cap G_2) \cdots P(G_N \mid G_1 \cap G_2 \cap \cdots \cap G_{N-1}) \\
&= 1 \cdot \frac{364}{365} \cdot \frac{363}{365} \cdots \frac{366-N}{365} \\
&= \frac{365\cdot 364 \cdot 363 \cdot \cdot ⋅ (366 - N)}{365^N}\\
&= \frac{365!}{365^N(366-N-1)!}\\
&= \frac{365!}{ 365^N(365-N)!}
\end{aligned}
$$

We now can obtain the formula for the desired probability:

$P(\text{at least 1 match}) = 1 - P(\text{no match})$

$P(\text{at least 1 match}) = 1 - \frac{365!}{ 365^N(365-N)!}$



### What is the tipping point?

Now that we have a formula, let's calculate some probabilities. Since $N$ appears inside a product of many terms, there's no clean algebraic way to isolate it directly. Instead, test increasing values of $N$ and determine when the probability of getting a match becomes more likely than not, that is: $P(\text{at least 1 match}) > 0.5$.

<center>

| N | P(at least 1 match) |
|---|---|
| 5 | 0.027 |
| 10 | 0.117 |
| 15 | 0.253 |
| 20 | 0.411 |
| 22 | 0.476 |
| 23 | **0.507** |

</center>

From the table, the smallest value of $N$ such that $P(\text{at least 1 match}) > 0.5$ is $N=23$. 23 is the tipping point!

It's a surprising result! In a group of only 23 people the probability of at least 2 people sharing a birthday is more likely than not. The reason it feels smaller than intuition suggests is that the number of *pairs* being compared grows much faster than the number of guests. With 23 people, there are $C(23, 2) = 23 \times 22 / 2 = 253$ unique pairs. Which is far more comparisons than most people imagine when they consider a room of 23.




## **Placing your Bet**

Remember your guess from the start? If you were near 23, you had better instincts than most of the chat. But there's a catch. 23 is where $P(\text{at least 1 match})$ first exceeds $0.5$, not necessarily the most likely value of N.

If you had to place a bet, would you choose 23 or something different? What would happen if we played Evan's game hundreds of times? Questions like this open the door to other measures of central tendency, like the **mode** (the most frequent value of $N$)  or the **expected value** (the average outcome across many repeated games). Both are concepts worth exploring another time!

*If you're curious, click the button below to run the game simulation 400 times, and imagine how these other measures of central tendency might compare.*


In [ ]:
run_400_button = widgets.Button(description="Run 400 Games!", layout=widgets.Layout(width='140px'))
run_400_output = widgets.Output()

def on_run_400_clicked(b):
    with run_400_output:
        run_400_output.clear_output()
        for _ in range(400):
            result, *_ = simulate_game()
            tally[result] += 1
        plot_tally()

run_400_button.on_click(on_run_400_clicked)
display(run_400_button, run_400_output)

Button(description='Run 400 Games!', layout=Layout(width='140px'), style=ButtonStyle())

Output()

## Additional Resources

- For a refresher on the measures of central tendency, check out this [source](https://opentextbc.ca/businesstechnicalmath/chapter/measures-of-central-tendency/) from BCcampus Open Publishing.

- For more background information and an extenstion of this problem, check out this [article](https://www.britannica.com/science/birthday-problem) from Britannica.